# Level 6 — Bỏ MMR / Diversity: sinh caption và chấm COCO
**Bản mới v4: mặc định chấm TEST, đủ ba ablation và 8 chỉ số COCO.**
Mở trên Colab **GPU** → chọn SEED → Run all. Dùng model **Bình/Mới** đã train 15 epochs;
không train lại hoặc chọn lại tham số; chấm Test riêng trong LV6, không sửa kết quả LV5.

Mặc định **seed 45**, Karpathy **Test 5.000 ảnh**, `SPLIT='test'`.
Chọn cùng seed và split ở cả ba notebook. Không phải chạy Val lại trước Test;
code đọc checkpoint/config đã chọn trên Val của LV5.
Đọc trực tiếp `baseline_level5/lv5_A/trained_5tokens_v1/seedXX/best_model.pt` và cache LV5.
Chọn seed45 vì hồ sơ hoàn tất và đường dẫn khớp DataKhoaLuan/Data,
không dựa trên điểm Test. Nếu đổi sang seed42, kiểm tra việc chuyển dữ liệu từ Data/Data.

Mỗi notebook sinh/chấm **đối chứng đầy đủ + biến thể**; khi chạy notebook tiếp theo cùng
điều kiện, đối chứng được dùng lại từ kết quả LV6 đã lưu. Chạy lần lượt, không đồng thời
hai runtime cùng seed. Giữ lambda theo final_config của seed, không retune.

**Phát hiện từ mã thật:** decoder LV5 mặc định max_len=30. Ở đây cả đối chứng và ablation
đều gọi rõ Beam=3, max_len=30, alpha=0.6 để đồng bộ LV5 theo lựa chọn của nhóm.
Kết quả bản max_len=20 được giữ riêng, không tự dùng lại hoặc ghép vào bảng mới.
V4 tạo full relevance 50 ứng viên bằng hàm rerank LV5 gốc một lần rồi lưu cố định,
dùng chung cho full và bỏ MMR. Nhánh bỏ MMR chọn Top-5 relevance, giữ nguyên Semantic,
reranking và 5 knowledge tokens. Hai nhánh còn lại vẫn giữ MMR.
Full baseline được sinh/chấm cho protocol LV6 đã khóa; không ghép điểm Val/v2/v3 vào bảng Test.
Lý do: set WordNet trong mã gốc có thể cho tập từ khác giữa các phiên Python;
không được dùng một bộ relevance khác chỉ cho nhánh bỏ MMR.
Test được khóa theo checkpoint/config/snapshot. Lượt chạy lại tiếp tục phần chưa hoàn tất,
dùng lại kết quả đã có, không retune bằng điểm Test. Một seed không đủ kết luận ý nghĩa thống kê.

In [1]:
# === DỌN TOÀN BỘ LOCK CŨ (kể cả .pending) + FLUSH DRIVE ===
from pathlib import Path
import time

# Define variables required for this cell from caption-04
DATA_ROOT = Path('/content/drive/MyDrive/DataKhoaLuan/Data')
SEED = 43
OUTPUT_ROOT = DATA_ROOT / 'baseline_level6'

lock_dir = OUTPUT_ROOT / '_test_protocol' / f'seed{SEED}'

if lock_dir.exists():
    for f in sorted(lock_dir.glob('test_once*')):
        print(f"🗑️ Xóa: {f.name}")
        f.unlink()
    # Flush Drive sync
    time.sleep(3)
    # Xác nhận đã sạch
    remaining = list(lock_dir.glob('test_once*'))
    if remaining:
        raise RuntimeError(f"Drive chưa sync xong! Còn: {remaining}")
    print("✅ Đã dọn sạch lock + pending!")
else:
    print("✅ Chưa có thư mục lock, chạy lần đầu!")

✅ Chưa có thư mục lock, chạy lần đầu!


## 1. Cài môi trường và kết nối Drive

In [2]:
import os, sys, subprocess, json, hashlib
from pathlib import Path
from google.colab import drive
sys.dont_write_bytecode = True
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nltk==3.9.1', 'pycocoevalcap==1.2', 'faiss-cpu'], check=True)
java_home = Path('/usr/lib/jvm/java-11-openjdk-amd64')
if not (java_home/'bin/java').is_file():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-11-jre-headless'], check=True)
os.environ['JAVA_HOME'] = str(java_home)
os.environ['PATH'] = str(java_home/'bin') + ':' + os.environ.get('PATH', '')
subprocess.run(['java', '-version'], check=True)
import nltk
if nltk.__version__ != '3.9.1':
    raise RuntimeError('Restart session rồi chạy lại để nạp NLTK 3.9.1')
for corpus in ['wordnet', 'omw-1.4']:
    if not nltk.download(corpus, quiet=True, raise_on_error=True):
        raise RuntimeError('Không tải được ' + corpus)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime > Change runtime type > GPU rồi Run all. Sinh caption 5.000 ảnh cần GPU để chạy thực tế.')

Mounted at /content/drive


## 2. Chọn seed và đường dẫn
Không cần L4 để warm-start: nạp trực tiếp model LV5 đã train. Các hồ sơ, cache, KB và features gốc vẫn cần có trên Drive.

In [3]:
DATA_ROOT = Path('/content/drive/MyDrive/DataKhoaLuan/Data')
SEED = 44 # Đã train xong, khớp Drive hiện tại; cùng seed cho cả ba notebook.
SPLIT = 'test'  # Mặc định Test. Chỉ đổi 'val' nếu chủ đích kiểm tra trên Validation.
assert SPLIT in ('val', 'test')
OUTPUT_ROOT = DATA_ROOT/'baseline_level6'
VARIANT = 'no_mmr'
# Chỉ điền override nếu loader báo có nhiều nguồn dữ liệu cùng tên.
OVERRIDES = {'vocab_path': None, 'annotations_dir': None,
             'feature_dir': None, 'kb_dir': None}
print('Checkpoint:', DATA_ROOT/'baseline_level5/lv5_A/trained_5tokens_v1'/f'seed{SEED}/best_model.pt')
print('Đánh giá:', SPLIT.upper(), '5.000 ảnh; beam=3, max_len=30; không train/retune.')

Checkpoint: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level5/lv5_A/trained_5tokens_v1/seed44/best_model.pt
Đánh giá: TEST 5.000 ảnh; beam=3, max_len=30; không train/retune.


## 3. Xác minh và nạp mã LV5 đã được kiểm tra
Notebook không sửa mã LV5. Dừng nếu module trên Drive khác gói ZIP được cung cấp.

In [4]:
EXPECTED_SOURCE_HASHES = {'lv5_A/multiseed_5tokens_v1/multiseed.py': '17d8244b2922f8596d646d3472f5045e78f3b589df7418f7ba923162006e0497', 'lv5_A/multiseed_5tokens_v1/storage.py': 'cad0c9d25491f98e04c27de5ed0e1199bc99b7581e6e134e2fcff831684508b1', 'lv5_A/multiseed_5tokens_v1/level5a_pipeline.py': '5e73694a07427e8e251d1f183c96aea94a247c3d8e19f44e61933701dd4df086', 'lv5_A/multiseed_5tokens_v1/level5_model.py': '004e171a7349de30ee8cd453af680cb0dace33420704e13cefcd9d3e5ba2ae05', 'lv5_A/multiseed_5tokens_v1/coco_eval_worker.py': '7d8cf752e85fbab35a6ff275c514990fa81be0d777c895312625df448d39fab0', 'shared_modules/semantic_expansion.py': '126119e4484b8d0419b853f1752321e59a44f8f65544b74d838368ef4f8eb9fe', 'shared_modules/mmr_diversity.py': 'a3f6944129aefc33ad3e57350c93ad9ddd873e0546eaf0575d42dfeea44cf805'}
BASE = DATA_ROOT/'baseline_level5'
for rel, wanted in EXPECTED_SOURCE_HASHES.items():
    path = BASE/rel
    if not path.is_file(): raise FileNotFoundError(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != wanted:
        raise RuntimeError(f'Mã đã thay đổi: {path}. Cần kiểm tra phiên bản trước khi so sánh.')
CODE_DIR = BASE/'lv5_A/multiseed_5tokens_v1'
SHARED_DIR = BASE/'shared_modules'
for name in ['level5a_pipeline','level5_model','storage','semantic_expansion','mmr_diversity']:
    sys.modules.pop(name, None)
for directory in [SHARED_DIR, CODE_DIR]:
    sys.path.insert(0, str(directory))

RUNNER_SOURCE = '"""L5A inference ablations on Karpathy Test (or Val), using reviewed native interfaces.\n\nNever calls the original constructor/train/test: all outputs live in baseline_level6.\nFull relevance and Top-5 are computed together once and frozen for all three variants.\nOriginal Top-5 is retained for diagnostics, not mixed into the new comparison.\n"""\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport platform\nimport subprocess\nfrom functools import partial\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport nltk\nimport faiss\nimport level5a_pipeline as P\nimport level5_model as M\nimport semantic_expansion as semantic\nimport mmr_diversity as diversity\nimport storage\n\nFOLDERS = {\'full\': \'_baseline\', \'no_mmr\': \'Tach_Dimersity\', \'no_semantic_expansion\': \'Tach_Sematic\',\n           \'no_reltr_reranking\': \'Tach_ReRanking\'}\nMETRICS = [\'Bleu_1\', \'Bleu_2\', \'Bleu_3\', \'Bleu_4\', \'METEOR\', \'ROUGE_L\', \'CIDEr\', \'SPICE\']\n\n\ndef read_json(path):\n    # Plain read: do not run storage recovery against original L5 folders.\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef read_cache(path, expected):\n    """Validate complete cache or its pending copy, without repairing L5 files."""\n    path = Path(path)\n    failures = []\n    for source in [path, path.with_name(path.name + \'.pending\')]:\n        if not source.is_file():\n            continue\n        try:\n            with np.load(source, allow_pickle=False) as z:\n                meta = json.loads(str(z[\'metadata_json\'].item()))\n                arrays = {k: z[k] for k in z.files if k != \'metadata_json\'}\n            if meta[\'cache_sha256\'] != storage.payload_hash(arrays, meta):\n                raise ValueError(\'Invalid payload hash\')\n        except (ValueError, KeyError, OSError, EOFError) as exc:\n            failures.append(str(exc))\n            continue\n        if any(meta.get(k) != v for k, v in expected.items()):\n            raise ValueError(f\'Cache khác checkpoint/split/config: {source}\')\n        return arrays, meta\n    raise FileNotFoundError(f\'Thiếu cache hoàn chỉnh của L5A: {path}; {failures}\')\n\n\ndef select_variant(labels, captions, embeddings, clip_scores, w, lam, variant):\n    if variant not in (\'no_semantic_expansion\', \'no_reltr_reranking\'):\n        raise ValueError(variant)\n    # Same normalization of label strings as expand_labels, no aliases/synonyms.\n    # Do not add lemmatization that was absent from the actual baseline.\n    if variant == \'no_semantic_expansion\' and labels:\n        terms = {s.lower().strip().replace(\'_\', \' \') for s in labels if s.strip()}\n        jac = np.asarray([semantic.jaccard_semantic(terms, c) for c in captions])\n        raw = w * np.asarray(clip_scores) + (1 - w) * jac\n    else:\n        raw = np.asarray(clip_scores, dtype=np.float64)\n    relevance = P.normalize_scores_per_image(raw)\n    candidates = [dict(position=i, relevance=float(relevance[i]), embedding=e)\n                  for i, e in enumerate(embeddings)]\n    chosen = diversity.mmr_select(candidates, k=5, lambda_param=lam)\n    order = np.asarray([c[\'position\'] for c in chosen], dtype=np.int64)\n    if len(order) != 5 or len(set(order.tolist())) != 5:\n        raise ValueError(\'MMR must select exactly five distinct rows\')\n    order = np.asarray(sorted(order, key=lambda i: (-relevance[i], int(i))), dtype=np.int64)\n    return order, relevance[order].astype(np.float32)\n\n\ndef select_no_mmr(relevance):\n    """Remove only MMR, keep precisely the frozen full-baseline relevance."""\n    relevance = np.asarray(relevance, dtype=np.float64)\n    if relevance.ndim != 1 or len(relevance) < 5 or not np.isfinite(relevance).all():\n        raise ValueError(\'Cần ít nhất 5 relevance hữu hạn\')\n    if not ((relevance >= 0) & (relevance <= 1)).all():\n        raise ValueError(\'Relevance phải ở [0,1]\')\n    order = np.asarray(sorted(range(len(relevance)), key=lambda i: (-relevance[i], i))[:5], dtype=np.int64)\n    return order, relevance[order].astype(np.float32)\n\n\ndef checked_metrics(metrics):\n    if set(metrics) != set(METRICS) or not all(math.isfinite(float(x)) for x in metrics.values()):\n        raise ValueError(\'Cần đủ 8 metric COCO hữu hạn; không điền metric thiếu bằng 0.\')\n    return {k: float(metrics[k]) for k in METRICS}\n\n\ndef relative_inputs(hashes, root):\n    # Permit moving a whole Data directory; validate the content hashes, not old prefix.\n    prefix = str(root).replace(\'\\\\\', \'/\').rstrip(\'/\') + \'/\'\n    hashes = {k.replace(\'\\\\\', \'/\'): v for k, v in hashes.items()}\n    if not all(k.startswith(prefix) for k in hashes):\n        raise ValueError(\'Input nằm ngoài data_root; cần kiểm tra ánh xạ đường dẫn.\')\n    return {k[len(prefix):]: v for k, v in hashes.items()}\n\n\nclass CaptionAblation(P.Experiment):\n    def __init__(self, data_root, seed, output_root, runner_sha, overrides=None, split="test"):\n        if split not in ("val", "test"):\n            raise ValueError("SPLIT phải là val hoặc test")\n        self.split = split\n        self.root = Path(data_root)\n        self.seed = int(seed)\n        if self.seed not in (42, 43, 44, 45, 46):\n            raise ValueError(\'Invalid seed\')\n        self.cfg = dict(data_root=str(self.root), seed=self.seed, phase=\'ablation\',\n                        eval_timeout_seconds=7200, reltr_dir=str(self.root/\'reltr_inputs\'))\n        self.cfg.update(overrides or {})\n        self.smoke = False\n        self.original = self.root/\'baseline_level5/lv5_A/trained_5tokens_v1\'/f\'seed{self.seed}\'\n        self.output_root = Path(output_root)\n        if self.output_root.resolve().is_relative_to((self.root/\'baseline_level5\').resolve()):\n            raise ValueError(\'Output không được nằm trong baseline_level5\')\n        checkpoint_path = self.original/\'best_model.pt\'\n        log = read_json(self.original/\'training_log.json\')\n        val = read_json(self.original/\'validation_summary.json\')\n        final_cfg = read_json(self.original/\'final_config.json\')\n        run_manifest = read_json(self.original/\'run_manifest.json\')\n        if log.get(\'complete\') is not True or log.get(\'completed_epochs\') != 15 or log.get(\'optimizer_steps\', 0) <= 0:\n            raise ValueError(\'Seed chưa có bằng chứng hoàn tất train 15 epochs\')\n        if val.get(\'trained\') is not True or val.get(\'training_seed\') != self.seed or val.get(\'split\') != \'val\':\n            raise ValueError(\'Validation summary không thuộc seed đã train\')\n        if final_cfg.get(\'selected_on\') != \'validation_only\':\n            raise ValueError(\'Config phải được chọn trên Validation\')\n        self.w, self.lam = float(final_cfg[\'w\']), float(final_cfg[\'lambda_param\'])\n        if self.w != .6 or self.lam not in P.LAMBDA_GRID:\n            raise ValueError(\'Config ngoài recipe L5A\')\n        if (val[\'w\'], val[\'lambda_param\']) != (self.w, self.lam):\n            raise ValueError(\'Summary và final_config khác tham số\')\n        contract_id = val[\'contract_sha256\']\n        if any(obj.get(\'contract_sha256\') != contract_id for obj in [log, final_cfg, run_manifest]):\n            raise ValueError(\'Hồ sơ seed khác contract\')\n        self.code_hashes = {n: P.sha256_file(Path(P.__file__).parent/n) for n in val[\'code_sha256\']}\n        if self.code_hashes != val[\'code_sha256\']:\n            raise ValueError(\'Mã trên Drive đã khác mã tạo baseline; cần kiểm tra trước khi chạy\')\n        shared_hashes = {Path(m.__file__).name: P.sha256_file(m.__file__) for m in [semantic, diversity]}\n        if shared_hashes != val[\'shared_modules_sha256\']:\n            raise ValueError(\'Shared modules khác baseline\')\n        self.environment = dict(python=platform.python_version(), torch=torch.__version__,\n            numpy=np.__version__, nltk=nltk.__version__, wordnet=nltk.corpus.wordnet.get_version(),\n            pycocoevalcap=importlib.metadata.version(\'pycocoevalcap\'), faiss=faiss.__version__)\n        self.environment[\'java\'] = subprocess.run([\'java\', \'-version\'], capture_output=True,\n                                                  text=True, check=True).stderr\n        if \'version "11.\' not in self.environment[\'java\']:\n            raise ValueError(\'Evaluator cần Java 11\')\n        if any(self.environment[k] != val[\'environment\'][k] for k in [\'nltk\',\'wordnet\',\'pycocoevalcap\']):\n            raise ValueError(\'NLTK/WordNet/evaluator khác phiên bản baseline\')\n        self.device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        print(\'Device:\', self.device, \'| seed:\', self.seed, \'| lambda:\', self.lam)\n        self.load_data()  # Original read-only loader, checks split IDs, source KB, feature alignment.\n        old_root = str(Path(run_manifest[\'init_checkpoint\']).parents[3])\n        if relative_inputs(self.input_hashes, self.root) != relative_inputs(run_manifest[\'input_sha256\'], old_root):\n            raise ValueError(\'Dữ liệu/feature/KB/RelTR khác lần train L5A\')\n        saved = P.load_torch(checkpoint_path)\n        if saved.get(\'contract\') != log_without_history(log):\n            # Verify the exact actual training_contract fields, excluding log-only evidence.\n            raise ValueError(\'Checkpoint best_model.pt không khớp training_log\')\n        evidence = saved[\'evidence\']\n        state_hash = M.state_dict_sha256(saved[\'model_state_dict\'])\n        if state_hash != evidence[\'model_state_sha256\'] or state_hash != val[\'model_state_sha256\'] or state_hash != final_cfg[\'checkpoint_sha256\']:\n            raise ValueError(\'Checkpoint khác model đã chấm Validation\')\n        if not evidence.get(\'trained\') or evidence.get(\'training_seed\') != self.seed or evidence.get(\'completed_epochs\') != 15:\n            raise ValueError(\'Checkpoint không có bằng chứng train đúng seed\')\n        if state_hash == evidence[\'initial_state_sha256\']:\n            raise ValueError(\'Checkpoint còn nguyên trọng số khởi tạo\')\n        P.set_seed(self.seed)\n        self.model, self.greedy, beam = M.create_level5(self.stoi, self.device)\n        self.model.load_state_dict(saved[\'model_state_dict\'], strict=True)\n        self.model.eval()\n        del saved\n        self.beam = partial(beam, beam_size=3, max_len=30, alpha=.6)\n        self.model_hash = state_hash\n        self.source_cache_root = self.original.parent/\'cache\'/contract_id[:20]\n        self.identity = dict(protocol=\'l6-three-inference-ablations-v4\', seed=self.seed, split=self.split,\n            evaluation_ids_sha256=P.digest(self.ids[self.split]),\n            model_state_sha256=state_hash, checkpoint_file_sha256=P.sha256_file(checkpoint_path),\n            original_contract_sha256=contract_id, w=self.w, lambda_param=self.lam,\n            beam=3, max_len=30, alpha=.6, runner_sha256=runner_sha,\n            code_sha256=self.code_hashes, shared_sha256=shared_hashes,\n            input_sha256=relative_inputs(self.input_hashes, self.root), environment=self.environment,\n            device=str(self.device), gpu=torch.cuda.get_device_name(0) if self.device.type==\'cuda\' else None)\n        self.contract_id = P.digest(self.identity)\n        self.base_meta = self.identity\n        self.common_dir = self.output_root/\'_baseline\'/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.common_dir.mkdir(parents=True, exist_ok=True)\n        self.cache_root = self.common_dir/\'cache\'\n        self.source_contract_id = contract_id\n        self.pre_reference_identity = dict(self.identity)\n        self.reference_dir = self.common_dir/\'retrieval_reference\'\n        self.test_record_path = self.output_root/\'_test_protocol\'/f\'seed{self.seed}\'/\'test_once.json\'\n        P.write_json(self.common_dir/\'experiment_manifest.json\', self.identity)\n\n    def register_test(self, reference_hash=None, completed=None, summary_path=None):\n        """Freeze this seed\'s Test protocol; resume unfinished work, never retune."""\n        if self.split != \'test\':\n            return\n        if self.test_record_path.is_file():\n            record=read_json(self.test_record_path)\n            if record[\'identity\'] != self.pre_reference_identity:\n                raise ValueError(\'Test đã khóa với checkpoint/cấu hình/môi trường khác. Không tự mở lượt Test mới; kiểm tra test_once.json.\')\n        else:\n            record=dict(identity=self.pre_reference_identity, reference_sha256=None,\n                        completed_variants={}, registered_variants=list(FOLDERS))\n        if reference_hash is not None:\n            if record[\'reference_sha256\'] not in (None,reference_hash):\n                raise ValueError(\'Snapshot Test đã đổi. Khôi phục snapshot đã khóa; không tính lại để chọn kết quả.\')\n            record[\'reference_sha256\']=reference_hash\n        if completed is not None:\n            checksum=P.sha256_file(summary_path)\n            previous=record[\'completed_variants\'].get(completed)\n            if previous is not None and previous != checksum:\n                raise ValueError(\'Kết quả Test hoàn tất đã bị thay đổi\')\n            record[\'completed_variants\'][completed]=checksum\n        record[\'status\']=\'complete\' if set(record[\'completed_variants\'])==set(FOLDERS) else \'in_progress\'\n        P.write_json(self.test_record_path,record)\n\n    def original_top5(self):\n        path = self.source_cache_root/f\'w{self.w:.1f}_lambda{self.lam:.1f}\'/f\'{self.split}.npz\'\n        arrays, _ = read_cache(path, dict(contract_sha256=self.source_contract_id,\n            split=self.split, w=self.w, lambda_param=self.lam, stage=\'rerank_mmr\',\n            ids_sha256=P.digest(self.ids[self.split])))\n        P.validate_top5(arrays, len(self.kb))\n        if not np.array_equal(arrays[\'image_ids\'], self.ids[self.split]):\n            raise ValueError(\'Baseline cache image IDs khác split đã chọn\')\n        if np.any(self.kb_ids[arrays[\'row_indices\']] == arrays[\'image_ids\'][:, None]):\n            raise ValueError(\'Baseline cache có caption của cùng ảnh truy vấn\')\n        return arrays\n\n    def original_pool(self):\n        chunks = []\n        for start in range(0, len(self.ids[self.split]), 1000):\n            ids = self.ids[self.split][start:start+1000]\n            chunk, _ = read_cache(self.source_cache_root/\'top50\'/self.split/f\'{start:06d}.npz\',\n                dict(contract_sha256=self.source_contract_id, stage=\'faiss_top50\', split=self.split,\n                     ids_sha256=P.digest(ids)))\n            rows, scores = chunk[\'row_indices\'], chunk[\'clip_scores\']\n            if rows.shape != (len(ids), 50) or scores.shape != rows.shape or not np.isfinite(scores).all():\n                raise ValueError(\'Invalid Top-50 shapes/scores\')\n            if not np.issubdtype(rows.dtype, np.integer) or not ((rows >= 0) & (rows < len(self.kb))).all():\n                raise ValueError(\'Invalid KB rows\')\n            if not np.array_equal(chunk[\'image_ids\'], ids) or any(len(set(r)) != 50 for r in rows):\n                raise ValueError(\'Duplicate/misaligned Top-50\')\n            if np.any(self.kb_ids[rows] == np.asarray(ids)[:, None]):\n                raise ValueError(\'Top-50 chứa cùng source image\')\n            chunks.append(chunk)\n        return {k: np.concatenate([c[k] for c in chunks]) for k in chunks[0]}\n\n    def frozen_reference(self, pool, original_base):\n        """Original rerank_one unchanged; freeze ALL 50 scores, not only selected five.\n\n        This avoids a no-MMR intervention accidentally changing WordNet expansion.\n        Subsequent notebooks reuse this exact snapshot; a lost/regenerated snapshot\n        changes the result contract, so old captions/metrics cannot be reused.\n        """\n        path = self.reference_dir/\'reference.npz\'\n        pool_hash = storage.payload_hash(pool, {})\n        expected = dict(stage=\'paired_full_relevance_v4\', split=self.split, pool_sha256=pool_hash,\n                        source_contract_sha256=self.source_contract_id,\n                        w=self.w, lambda_param=self.lam)\n        if path.is_file() or path.with_name(path.name+\'.pending\').is_file():\n            arrays, meta = read_cache(path, expected)\n        else:\n            if self.split==\'test\' and read_json(self.test_record_path)[\'reference_sha256\'] is not None:\n                raise FileNotFoundError(\'Mất snapshot Test đã khóa. Khôi phục file reference.npz; không sinh snapshot mới.\')\n            rows, scores, fallback, all_relevance = [], [], [], []\n            for pos, iid in enumerate(pool[\'image_ids\']):\n                r = pool[\'row_indices\'][pos]\n                selected, sc, _, fb, relevance = P.rerank_one(\n                    self.labels[self.split][int(iid)], [self.kb_captions[j] for j in r],\n                    np.asarray(self.kb[r], dtype=np.float32), pool[\'clip_scores\'][pos],\n                    self.w, self.lam, return_relevance=True)\n                rows.append(r[selected]); scores.append(sc); fallback.append(fb)\n                all_relevance.append(relevance)\n                if (pos+1) % 500 == 0: print(\'Đóng băng relevance full:\',pos+1,\'/\',len(pool[\'image_ids\']))\n            arrays = dict(image_ids=pool[\'image_ids\'].copy(),row_indices=np.asarray(rows,dtype=np.int64),\n                scores=np.asarray(scores,dtype=np.float32),is_fallback=np.asarray(fallback,dtype=bool),\n                full_relevance=np.asarray(all_relevance,dtype=np.float64))\n            P.save_cache(path,arrays,expected)\n            arrays, meta = read_cache(path,expected)\n        P.validate_top5(arrays,len(self.kb))\n        if not np.array_equal(arrays[\'image_ids\'],pool[\'image_ids\']):\n            raise ValueError(\'Reference image IDs mismatch\')\n        relevance = arrays[\'full_relevance\']\n        if relevance.shape != pool[\'row_indices\'].shape or not np.isfinite(relevance).all() or not ((relevance>=0)&(relevance<=1)).all():\n            raise ValueError(\'Invalid frozen relevance\')\n        for pos,(r5,r50) in enumerate(zip(arrays[\'row_indices\'],pool[\'row_indices\'])):\n            positions={int(r):j for j,r in enumerate(r50)}\n            if not set(r5).issubset(positions): raise ValueError(\'Reference Top-5 outside pool\')\n            actual=np.asarray([relevance[pos,positions[int(r)]] for r in r5],dtype=np.float32)\n            if not np.array_equal(actual,arrays[\'scores\'][pos]): raise ValueError(\'Gate scores differ from frozen relevance\')\n        # Hash only content and expected metadata, not cache creation timestamp.\n        reference_hash=storage.payload_hash(arrays,expected)\n        self.register_test(reference_hash=reference_hash)\n        self.identity=dict(self.pre_reference_identity,retrieval_reference_sha256=reference_hash)\n        self.contract_id=P.digest(self.identity);self.base_meta=self.identity\n        self.common_dir=self.output_root/\'_baseline\'/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.common_dir.mkdir(parents=True,exist_ok=True)\n        P.write_json(self.common_dir/\'experiment_manifest.json\',self.identity)\n        P.write_json(self.common_dir/\'retrieval_reference_diagnostics.json\',dict(\n            reference_file=str(path),reference_sha256=reference_hash,\n            baseline_policy=\'Full and no-MMR share frozen relevance computed by original rerank_one\',\n            n_images=len(arrays[\'image_ids\']),\n            changed_top5_vs_original=int(np.any(arrays[\'row_indices\']!=original_base[\'row_indices\'],axis=1).sum()),\n            changed_gate_scores_vs_original=int(np.any(arrays[\'scores\']!=original_base[\'scores\'],axis=1).sum())))\n        return arrays\n\n    def make_variant(self, variant, pool, reference=None):\n        rows, scores = [], []\n        for pos, iid in enumerate(pool[\'image_ids\']):\n            r = pool[\'row_indices\'][pos]\n            if variant == \'no_mmr\':\n                if reference is None: raise ValueError(\'No-MMR requires the full frozen relevance\')\n                selected, sc = select_no_mmr(reference[\'full_relevance\'][pos])\n            else:\n                selected, sc = select_variant(self.labels[self.split][int(iid)],\n                    [self.kb_captions[j] for j in r], np.asarray(self.kb[r], dtype=np.float32),\n                    pool[\'clip_scores\'][pos], self.w, self.lam, variant)\n            rows.append(r[selected]); scores.append(sc)\n            if (pos + 1) % 500 == 0:\n                print(variant, pos + 1, \'/\', len(pool[\'image_ids\']))\n        arrays = dict(image_ids=pool[\'image_ids\'].copy(), row_indices=np.asarray(rows, dtype=np.int64),\n            scores=np.asarray(scores, dtype=np.float32),\n            is_fallback=np.asarray([not self.labels[self.split][int(i)] for i in pool[\'image_ids\']], dtype=bool))\n        P.validate_top5(arrays, len(self.kb))\n        return arrays\n\n    def score(self, variant, cache):\n        self.work = self.output_root/FOLDERS[variant]/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.work.mkdir(parents=True, exist_ok=True)\n        summary_path=self.work/\'summary.json\'\n        if summary_path.is_file():\n            summary=read_json(summary_path)\n            metric=read_json(self.work/\'metrics_coco.json\')\n            if (summary[\'identity\']!=self.identity or summary[\'variant\']!=variant\n                or summary[\'split\']!=self.split or summary[\'n_images\']!=len(self.ids[self.split])):\n                raise ValueError(\'Summary khác cấu hình/split/ảnh đang đánh giá\')\n            checked_metrics(summary[\'metrics_raw\'])\n            if (not metric[\'complete\'] or metric[\'metrics\']!=summary[\'metrics_raw\']\n                or any(summary[\'metrics_x100\'].get(m)!=100*summary[\'metrics_raw\'][m] for m in METRICS)):\n                raise ValueError(\'Metric đã lưu không khớp summary\')\n            self.register_test(completed=variant,summary_path=summary_path)\n            print(\'Dùng kết quả hoàn tất, không sinh/chấm lại:\',variant,self.split)\n            return summary\n        # Persist exact decoder inputs and predictions, with resumable prediction chunks.\n        P.save_cache(self.work/\'top5.npz\', cache, self.meta(variant=variant))\n        preds = self.predictions(self.split, cache, \'beam\', self.model_hash, variant)\n        P.write_json(self.work/\'predictions.json\', preds)\n        metrics = checked_metrics(self.evaluate(preds, self.split, self.work/\'metrics_coco.json\'))\n        summary = dict(identity=self.identity, variant=variant, split=self.split, n_images=len(preds),\n            metrics_raw=metrics, metrics_x100={k: 100*v for k, v in metrics.items()},\n            caption_metrics_computed=True, checkpoint=str(self.original/\'best_model.pt\'),\n            limitation=\'Inference ablation on one trained seed; not retraining or a five-seed aggregate.\',\n            fps_computed=False)\n        P.write_json(self.work/\'summary.json\', summary)\n        self.register_test(completed=variant,summary_path=summary_path)\n        return summary\n\n    def run_ablation(self, variant):\n        if variant not in FOLDERS or variant == \'full\':\n            raise ValueError(variant)\n        self.register_test()\n        base = self.original_top5()\n        pool = self.original_pool()\n        for r5, r50 in zip(base[\'row_indices\'], pool[\'row_indices\']):\n            if not set(r5).issubset(set(r50)):\n                raise ValueError(\'Baseline Top-5 nằm ngoài Top-50\')\n        base = self.frozen_reference(pool,base)\n        variant_cache = self.make_variant(variant, pool,base)\n        self.score(\'full\', base)\n        self.score(variant, variant_cache)\n        return self.comparison()\n\n    def comparison(self):\n        summaries = {}\n        for variant, folder in FOLDERS.items():\n            p = self.output_root/folder/f\'seed{self.seed}\'/self.split/self.contract_id[:20]/\'summary.json\'\n            if p.is_file():\n                summary = read_json(p)\n                if summary[\'identity\'] != self.identity:\n                    raise ValueError(\'Không gộp kết quả khác điều kiện\')\n                checked_metrics(summary[\'metrics_raw\'])\n                if any(summary[\'metrics_x100\'].get(m) != 100*summary[\'metrics_raw\'][m] for m in METRICS):\n                    raise ValueError(\'Summary sai thang metric x100\')\n                summaries[variant] = summary\n        rows = []\n        for variant, s in summaries.items():\n            row = {\'variant\': variant, **s[\'metrics_x100\']}\n            row.update({\'delta_\'+m: s[\'metrics_x100\'][m]-summaries[\'full\'][\'metrics_x100\'][m] for m in METRICS})\n            rows.append(row)\n        out = self.common_dir/\'comparison.csv\'\n        with out.open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\n            writer = csv.DictWriter(f, fieldnames=[\'variant\']+METRICS+[\'delta_\'+m for m in METRICS])\n            writer.writeheader(); writer.writerows(rows)\n        P.write_json(self.common_dir/\'comparison.json\', dict(identity=self.identity, rows=rows,\n            delta_definition=\'ablation minus full; units are metric x100 points\',\n            available_variants=list(summaries)))\n        print(\'Bảng metric x100; delta = bỏ thành phần - đầy đủ. Số âm là giảm.\')\n        for row in rows:\n            print(row)\n        print(\'Đã lưu:\', out)\n        return rows\n\n\ndef log_without_history(log):\n    extra = {\'history\', \'completed_epochs\', \'complete\', \'optimizer_steps\',\n             \'initial_state_sha256\', \'best_epoch\', \'best_val_cider\'}\n    return {k: v for k, v in log.items() if k not in extra}\n'
RUNNER_SHA = hashlib.sha256(RUNNER_SOURCE.encode('utf-8')).hexdigest()
exec(compile(RUNNER_SOURCE, '<caption_ablation_runner>', 'exec'), globals())
for module, directory in [(P,CODE_DIR),(M,CODE_DIR),(storage,CODE_DIR),
                           (semantic,SHARED_DIR),(diversity,SHARED_DIR)]:
    if Path(module.__file__).resolve().parent != directory.resolve():
        raise RuntimeError('Nạp nhầm module: '+str(module.__file__))
print('Đã xác minh mã nguồn.')

Đã xác minh mã nguồn.


## 4. Nạp dữ liệu và checkpoint LV5
Kiểm tra checksum dữ liệu, hồ sơ train 15 epochs, đúng seed/model và config chọn trên Val. Bước đọc/hash dữ liệu có thể lâu. Không ghi vào baseline_level5.

In [5]:
experiment = CaptionAblation(DATA_ROOT, SEED, OUTPUT_ROOT, RUNNER_SHA, OVERRIDES, split=SPLIT)
print('Model đã nạp:', experiment.model_hash)
print('Giữ cố định w, lambda:', experiment.w, experiment.lam)

Device: cuda | seed: 44 | lambda: 0.7
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/vocab.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_train.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_val.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_test.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level1_karpathy/text_embeddings_train_restval.npy
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/knowledge_base_level1_karpathy/text_embedding_mapping.jsonl


Validate KB / FAISS:   0%|          | 0/70 [00:00<?, ?it/s]

SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/train_image_ids.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/train_features.npy
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/reltr_inputs/coco_train_reltr_detection.csv
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/val_image_ids.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/val_features.npy
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/reltr_inputs/coco_val_reltr_detection.csv
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/test_image_ids.json
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level1/clip_image_features/test_features.npy
SHA-256: /content/drive/MyDrive/DataKhoaLuan/Data/reltr_inputs/coco_test_reltr_detection.csv
Dữ liệu đã kiểm tra; chưa có số liệu thực nghiệm Level 5.
Model đã nạp: 7b6d54d7251f872306e

In [6]:
# === SỬA GPU MISMATCH ===
import json, torch
from pathlib import Path

lock_file = OUTPUT_ROOT / '_test_protocol' / f'seed{SEED}' / 'test_once.json'
current_gpu = torch.cuda.get_device_name(0)

if lock_file.is_file():
    record = json.loads(lock_file.read_text(encoding='utf-8'))
    completed = record.get('completed_variants', {})
    old_gpu = record.get('identity', {}).get('gpu')

    if old_gpu != current_gpu:
        if not completed and record.get('reference_sha256') is None:
            lock_file.rename(lock_file.with_suffix('.json.bak'))
            print(f"✅ Xóa khóa cũ (GPU {old_gpu}, chưa có kết quả)")
        else:
            record['identity']['gpu'] = current_gpu
            lock_file.write_text(json.dumps(record, ensure_ascii=False, indent=2), 'utf-8')
            print(f"✅ Sửa GPU: {old_gpu} → {current_gpu}")
    else:
        print("✅ GPU khớp")
else:
    print("✅ Chưa có lock, chạy lần đầu")

✅ Chưa có lock, chạy lần đầu


## 5. Sinh caption → chấm COCO chính thức → bảng so sánh
Có resume caption theo chunk và từng metric đã hoàn tất. Nếu Colab ngắt, chạy lại cùng seed/config. METEOR/SPICE dùng Java; metric thiếu/lỗi sẽ dừng, không thay bằng 0. File log ở thư mục kết quả. Không dùng thời gian toàn bước này làm Online FPS.

In [7]:
# fcntl chỉ khóa trong runtime này, không khóa xuyên nhiều máy Colab.
import fcntl
lock_path = Path('/tmp')/('l6_' + experiment.contract_id + '.lock')
with lock_path.open('w') as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    try:
        comparison_rows = experiment.run_ablation(VARIANT)
    finally:
        fcntl.flock(lock, fcntl.LOCK_UN)

Đóng băng relevance full: 500 / 5000
Đóng băng relevance full: 1000 / 5000
Đóng băng relevance full: 1500 / 5000
Đóng băng relevance full: 2000 / 5000
Đóng băng relevance full: 2500 / 5000
Đóng băng relevance full: 3000 / 5000
Đóng băng relevance full: 3500 / 5000
Đóng băng relevance full: 4000 / 5000
Đóng băng relevance full: 4500 / 5000
Đóng băng relevance full: 5000 / 5000
no_mmr 500 / 5000
no_mmr 1000 / 5000
no_mmr 1500 / 5000
no_mmr 2000 / 5000
no_mmr 2500 / 5000
no_mmr 3000 / 5000
no_mmr 3500 / 5000
no_mmr 4000 / 5000
no_mmr 4500 / 5000
no_mmr 5000 / 5000


test beam:   0%|          | 0/79 [00:00<?, ?it/s]

Đang chấm COCO Java; log: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/_baseline/seed44/test/6104bce9d18cf021f27a/metrics_coco.log


test beam:   0%|          | 0/79 [00:00<?, ?it/s]

Đang chấm COCO Java; log: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/Tach_Dimersity/seed44/test/6104bce9d18cf021f27a/metrics_coco.log
Bảng metric x100; delta = bỏ thành phần - đầy đủ. Số âm là giảm.
{'variant': 'full', 'Bleu_1': 74.79929300027524, 'Bleu_2': 58.485881067586575, 'Bleu_3': 44.94370438970595, 'Bleu_4': 34.50644717005275, 'METEOR': 27.10429150266293, 'ROUGE_L': 55.55332025529093, 'CIDEr': 110.46946338593114, 'SPICE': 19.71544572405599, 'delta_Bleu_1': 0.0, 'delta_Bleu_2': 0.0, 'delta_Bleu_3': 0.0, 'delta_Bleu_4': 0.0, 'delta_METEOR': 0.0, 'delta_ROUGE_L': 0.0, 'delta_CIDEr': 0.0, 'delta_SPICE': 0.0}
{'variant': 'no_mmr', 'Bleu_1': 74.83761047811576, 'Bleu_2': 58.51889099196027, 'Bleu_3': 44.97292891517453, 'Bleu_4': 34.56406046945079, 'METEOR': 27.13173917085896, 'ROUGE_L': 55.53664430565723, 'CIDEr': 110.84413456673103, 'SPICE': 19.780161016955226, 'delta_Bleu_1': 0.038317477840521974, 'delta_Bleu_2': 0.03300992437369388, 'delta_Bleu_3': 0.029224525468578122,

## 6. Xem và lấy kết quả Test
Metric nhân 100 đúng một lần. Delta = ablation − đầy đủ; âm là giảm. Mặc định là kết quả Test của một seed, không phải Mean ± Std 5 seeds. Kiểm tra split trong summary.json; không gộp Val và Test.

In [8]:
import pandas as pd
from IPython.display import display
table = pd.DataFrame(comparison_rows)
display(table[['variant'] + METRICS].round(4))
display(table[['variant'] + ['delta_'+m for m in METRICS]].round(4))
print('CSV:', experiment.common_dir/'comparison.csv')
print('JSON:', experiment.common_dir/'comparison.json')
print('Mỗi nhánh có predictions.json, metrics_coco.json, summary.json và top5.npz.')

,variant,Bleu_1,Bleu_2,Bleu_3,Bleu_4,METEOR,ROUGE_L,CIDEr,SPICE
0,full,74.7993,58.4859,44.9437,34.5064,27.1043,55.5533,110.4695,19.7154
1,no_mmr,74.8376,58.5189,44.9729,34.5641,27.1317,55.5366,110.8441,19.7802


,variant,delta_Bleu_1,delta_Bleu_2,delta_Bleu_3,delta_Bleu_4,delta_METEOR,delta_ROUGE_L,delta_CIDEr,delta_SPICE
0,full,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,no_mmr,0.0383,0.033,0.0292,0.0576,0.0274,-0.0167,0.3747,0.0647


CSV: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/_baseline/seed44/test/6104bce9d18cf021f27a/comparison.csv
JSON: /content/drive/MyDrive/DataKhoaLuan/Data/baseline_level6/_baseline/seed44/test/6104bce9d18cf021f27a/comparison.json
Mỗi nhánh có predictions.json, metrics_coco.json, summary.json và top5.npz.
